# Lab 7: Meeting Summarizer Agent

**AI Agent Lab 7 of 8**

---

## What this lab does

In this lab you will build a Meeting Summarizer Agent. You give it a realistic meeting transcript (a multi-line string with speaker labels and dialogue). The agent extracts the list of attendees, identifies action items (lines with words like 'will', 'deadline', or 'by Friday'), and identifies decisions made in the meeting. It prints a clean formatted summary. As a bonus, it reuses a simplified version of the email writer from Lab 1 to turn the summary into a follow-up email.

## What you will learn

**Python concepts:**
- How to process a multi-line string line by line
- How to extract information using `re.search()` and `re.findall()`
- How to build sets to collect unique values (unique attendees)
- How to build lists by filtering lines based on conditions
- How to write reusable functions that call other functions

**Agent concepts:**
- Information extraction as a core AI task
- Rule-based entity extraction (names, action items, decisions)
- Document summarisation without an LLM
- Composing outputs: summary -> follow-up email

## Important things to know before starting

- **What is information extraction?** It means pulling structured data (names, dates, tasks) out of unstructured text (like a transcript). This is one of the most common NLP tasks.
- **What is a set?** A set is like a list but it only keeps unique values. We use one to collect attendee names without duplicates.
- **What is a transcript?** A transcript is a written record of a spoken conversation, with each line showing who spoke and what they said.
- **This lab runs completely offline.**

## How the agent works

```
Input: meeting transcript (multi-line string)
        |
        v
  Extract: attendees (speaker names)
  Extract: action items (lines with will/deadline/by Friday etc.)
  Extract: decisions (lines with decided/agreed/confirmed etc.)
        |
        v
  Format: print a structured meeting summary
        |
        v
  Compose: generate a follow-up email from the summary
```

---
## Step 0: Libraries and Setup

This lab uses **only built-in Python libraries**.

We use:
- `re` - for extracting speaker names and keywords from the transcript
- `datetime` - for adding a date to the follow-up email
- `textwrap` - for wrapping long lines in the email output
- `sys` - to print the Python version

In [ ]:
# No installation needed
# Optional: # !pip install nothing

import sys
print('Setup check passed!')
print(f'Python version: {sys.version}')

---
## Importing Libraries

| Library | What it is used for in this lab |
|---|---|
| `re` | Extracts speaker names and keyword-based lines from the transcript |
| `datetime` | Adds a date to the follow-up email |
| `textwrap` | Wraps long lines in the email output to 70 characters |

In [ ]:
import re        # used to extract names and keywords from the transcript
import datetime # used to add a date to the follow-up email
import textwrap # used to wrap long lines in the email output

print('Libraries imported successfully.')

---
## Step 1: Define the Meeting Transcript

**What this block does:** We define a multi-line string called `meeting_transcript_text`. It simulates a 15-line meeting transcript with four speakers, decisions, action items, and a deadline.

**Why we need it:** This is the raw input the agent will process. In a real system, this could come from a meeting recording transcription service.

**What to expect as output:** A confirmation message showing how many lines are in the transcript.

In [ ]:
# The meeting transcript. Each line follows the format 'SpeakerName: dialogue text'.
# Action items use words like 'will', 'deadline', or 'by Friday'.
# Decisions use words like 'decided', 'agreed', or 'confirmed'.
meeting_transcript_text = """
Sarah: Good morning everyone, let us get started. Today we are reviewing the product launch timeline.
James: Thanks Sarah. We have confirmed the product name as 'Nexus Pro'.
Priya: Agreed. Marketing agreed to begin the social media campaign by next Monday.
David: I will prepare the final budget report and send it to Sarah by Friday.
Sarah: Great. We decided to move the launch date to the 15th of next month.
James: Noted. I will coordinate with the design team to finalise the packaging by Wednesday.
Priya: I will write the press release and have it ready for review by Thursday.
David: Also, we need to confirm the vendor contract. The deadline is end of this week.
Sarah: James, can you handle that vendor call?
James: Yes, I will set up the call with the vendor on Tuesday morning.
Priya: The QA team confirmed that testing is on track and will finish by next Friday.
David: Good. We agreed that no new features will be added after today's meeting.
Sarah: Confirmed. Post-launch support plan will be shared by David next Monday.
Priya: One more thing - we decided the product demo will be recorded, not live.
Sarah: Perfect. Let us wrap up. Everyone please action your items before the next meeting.
"""

# Count how many non-empty lines are in the transcript
all_transcript_lines = [line.strip() for line in meeting_transcript_text.split('\n') if line.strip()]
print(f'Transcript loaded: {len(all_transcript_lines)} non-empty lines.')

---
## Step 2: Build the Attendee Extractor

**What this block does:** We define `extract_attendees_from_transcript()`. It reads the transcript line by line, looks for the pattern 'SpeakerName: text' (a name followed by a colon), and collects all unique names.

**Why we need it:** Knowing who attended the meeting is the first piece of information in any meeting summary.

**What to expect as output:** The function is defined and immediately tested.

In [ ]:
def extract_attendees_from_transcript(transcript_text):
    """
    Extracts a sorted list of unique attendee names from a transcript.

    Looks for lines starting with a capitalised name followed by a colon,
    for example: 'Sarah: Good morning...'

    Takes in:
      transcript_text - the full meeting transcript as a multi-line string

    Returns:
      A sorted list of unique attendee name strings.
    """

    set_of_attendee_names = set()  # using a set to avoid duplicates automatically

    for single_line in transcript_text.split('\n'):
        stripped_line = single_line.strip()
        # Look for the pattern: one or more capitalised words followed by a colon
        name_match = re.match(r'^([A-Z][a-z]+(?:\s[A-Z][a-z]+)*):', stripped_line)
        if name_match:
            extracted_name = name_match.group(1)
            set_of_attendee_names.add(extracted_name)

    return sorted(list(set_of_attendee_names))


# Test the extractor
extracted_attendees = extract_attendees_from_transcript(meeting_transcript_text)
print(f'Attendees found ({len(extracted_attendees)}): {extracted_attendees}')

---
## Step 3: Build the Action Item Extractor

**What this block does:** We define `extract_action_items_from_transcript()`. It loops over the transcript and collects lines that contain action-item keywords like 'will', 'deadline', 'by Friday', 'by Wednesday', 'by Thursday', etc.

**Why we need it:** Action items are the most important output of any meeting - they tell people what they need to do. This is why summarisers focus on them.

**What to expect as output:** The function is tested and a list of action items is printed.

In [ ]:
# Keywords that indicate an action item
action_item_trigger_words = [
    'will ', 'deadline', 'by friday', 'by monday', 'by tuesday',
    'by wednesday', 'by thursday', 'by next', 'action', 'need to', 'must'
]


def extract_action_items_from_transcript(transcript_text):
    """
    Extracts lines from the transcript that contain action item keywords.

    Takes in:
      transcript_text - the full meeting transcript (multi-line string)

    Returns:
      A list of strings, each being one action item line from the transcript.
    """

    list_of_action_item_lines = []

    for single_line in transcript_text.split('\n'):
        stripped_line = single_line.strip()
        if not stripped_line:
            continue  # skip empty lines

        # Check if any action keyword appears in this line (case-insensitive)
        lowered_line = stripped_line.lower()
        for trigger_word in action_item_trigger_words:
            if trigger_word in lowered_line:
                list_of_action_item_lines.append(stripped_line)
                break  # only add the line once even if multiple keywords match

    return list_of_action_item_lines


# Test the extractor
extracted_action_items = extract_action_items_from_transcript(meeting_transcript_text)
print(f'Action items found: {len(extracted_action_items)}')
for item in extracted_action_items:
    print(f'  - {item}')

---
## Step 4: Build the Decision Extractor

**What this block does:** We define `extract_decisions_from_transcript()`. Similar to action items, it collects lines containing decision keywords: 'decided', 'agreed', 'confirmed', 'approved', 'resolved'.

**Why we need it:** Decisions are what the team agreed to do at the strategic level, as opposed to action items which are individual tasks.

**What to expect as output:** A list of decision lines from the transcript.

In [ ]:
# Keywords that indicate a decision was made
decision_trigger_words = ['decided', 'agreed', 'confirmed', 'approved', 'resolved', 'concluded']


def extract_decisions_from_transcript(transcript_text):
    """
    Extracts lines from the transcript that contain decision keywords.

    Takes in:
      transcript_text - the full meeting transcript (multi-line string)

    Returns:
      A list of strings, each being one decision line from the transcript.
    """

    list_of_decision_lines = []

    for single_line in transcript_text.split('\n'):
        stripped_line = single_line.strip()
        if not stripped_line:
            continue

        lowered_line = stripped_line.lower()
        for trigger_word in decision_trigger_words:
            if trigger_word in lowered_line:
                list_of_decision_lines.append(stripped_line)
                break

    return list_of_decision_lines


# Test the extractor
extracted_decisions = extract_decisions_from_transcript(meeting_transcript_text)
print(f'Decisions found: {len(extracted_decisions)}')
for decision in extracted_decisions:
    print(f'  - {decision}')

---
## Step 5: Build the Meeting Summarizer Agent

**What this block does:** We define `meeting_summarizer_agent()`. It calls all three extractors, then formats and prints a complete meeting summary.

**Why we need it:** This is the main agent that ties everything together. It calls the extractors, organises the results, and presents them clearly.

**What to expect as output:** A nicely formatted meeting summary with three sections.

In [ ]:
def meeting_summarizer_agent(transcript_text, meeting_title='Team Meeting'):
    """
    The main Meeting Summarizer Agent.

    Extracts attendees, action items, and decisions from a transcript,
    then prints a formatted meeting summary.

    Takes in:
      transcript_text - the meeting transcript (multi-line string)
      meeting_title   - a label for the meeting (string, optional)

    Returns:
      A dictionary containing the extracted data for use by other functions.
    """

    todays_date = datetime.date.today().strftime('%B %d, %Y')

    # Extract all three types of information
    attendees_list = extract_attendees_from_transcript(transcript_text)
    action_items_list = extract_action_items_from_transcript(transcript_text)
    decisions_list = extract_decisions_from_transcript(transcript_text)

    # Print the formatted summary
    print('=' * 65)
    print(f'MEETING SUMMARY: {meeting_title}')
    print(f'Date: {todays_date}')
    print('=' * 65)

    print(f'ATTENDEES ({len(attendees_list)}):')
    for attendee_name in attendees_list:
        print(f'  - {attendee_name}')

    print(f'\nDECISIONS ({len(decisions_list)}):')
    if decisions_list:
        for decision_line in decisions_list:
            print(f'  - {decision_line}')
    else:
        print('  No decisions recorded.')

    print(f'\nACTION ITEMS ({len(action_items_list)}):')
    if action_items_list:
        for action_item_line in action_items_list:
            print(f'  - {action_item_line}')
    else:
        print('  No action items recorded.')

    print('=' * 65)

    # Return the extracted data as a dictionary for use by the email generator
    return {
        'meeting_title': meeting_title,
        'date': todays_date,
        'attendees': attendees_list,
        'decisions': decisions_list,
        'action_items': action_items_list
    }


# Run the summarizer on our meeting transcript
meeting_summary_data = meeting_summarizer_agent(
    meeting_transcript_text,
    meeting_title='Product Launch Planning Meeting'
)

---
## Step 6: Generate a Follow-Up Email from the Summary

**What this block does:** We define `generate_followup_email_from_summary()`. This reusable function takes the summary dictionary returned by the agent and formats it into a professional follow-up email. This reuses the email template idea from Lab 1.

**Why we need it:** Follow-up emails after meetings are extremely common. Showing how the summarizer output feeds directly into an email writer demonstrates how agents can be composed into pipelines.

**What to expect as output:** A complete follow-up email based on the meeting summary.

In [ ]:
# This function reuses the email-writing idea from Lab 1 but fills in the
# content from the structured meeting summary rather than a template.

def generate_followup_email_from_summary(summary_dictionary, sender_name, recipient_names_list):
    """
    Generates a follow-up email from a meeting summary dictionary.

    Takes in:
      summary_dictionary   - the dictionary returned by meeting_summarizer_agent()
      sender_name          - the name of the person sending the follow-up (string)
      recipient_names_list - a list of recipient name strings

    Returns:
      The formatted email text as a string.
    """

    # Build the recipient greeting
    if len(recipient_names_list) == 1:
        greeting_line = f'Dear {recipient_names_list[0]},'
    else:
        greeting_line = f'Dear {" and ".join(recipient_names_list)},'

    # Build the attendees line
    attendees_formatted = ', '.join(summary_dictionary['attendees'])

    # Build the decisions section
    if summary_dictionary['decisions']:
        decisions_text = '\n'.join(f'  - {d}' for d in summary_dictionary['decisions'])
    else:
        decisions_text = '  No decisions recorded.'

    # Build the action items section
    if summary_dictionary['action_items']:
        action_items_text = '\n'.join(f'  - {a}' for a in summary_dictionary['action_items'])
    else:
        action_items_text = '  No action items recorded.'

    # Assemble the full email
    email_text = (
        f'Subject: Follow-Up - {summary_dictionary["meeting_title"]} ({summary_dictionary["date"]})\n\n'
        f'{greeting_line}\n\n'
        f'Thank you for attending the {summary_dictionary["meeting_title"]} on '
        f'{summary_dictionary["date"]}. This email summarises what was discussed.\n\n'
        f'Attendees: {attendees_formatted}\n\n'
        f'Key Decisions:\n{decisions_text}\n\n'
        f'Action Items:\n{action_items_text}\n\n'
        f'Please action your items before our next meeting. '
        f'Do not hesitate to reach out if you have any questions.\n\n'
        f'Kind regards,\n'
        f'{sender_name}'
    )

    return email_text


# Generate and print the follow-up email from our meeting summary
followup_email_text = generate_followup_email_from_summary(
    summary_dictionary=meeting_summary_data,
    sender_name='Sarah',
    recipient_names_list=['James', 'Priya', 'David']
)

print('GENERATED FOLLOW-UP EMAIL')
print('=' * 65)
# Wrap lines for readability
for single_line in followup_email_text.split('\n'):
    print(textwrap.fill(single_line, width=65) if single_line else '')
print('=' * 65)

---
## Summary

You built a **Meeting Summarizer Agent** that extracts structured information from a raw transcript and generates a follow-up email.

### Key takeaways
- **Multi-line string processing:** Splitting on newlines and iterating line by line is a simple and powerful text analysis technique.
- **Sets for unique collections:** Using a set to collect attendee names automatically handles duplicates.
- **Composability:** The summarizer returns a dictionary that the email generator uses as input. This is the concept of composing agents into pipelines.
- **Keyword extraction mirrors NLP:** Real NLP systems use trained models to find action items and decisions, but the output is the same type of labelled list.
- **Reusable functions:** `generate_followup_email_from_summary()` is completely independent of the summarizer - it just needs the right dictionary shape.

### Connection to the next lab

In **Lab 8 (Capstone)**, you will combine everything you have learned into a multi-agent pipeline: a Triage Agent, a Writer Agent, and a Sender Agent working together in sequence.

---

## Try It Yourself

1. **Edit the transcript:** Add three more lines to `meeting_transcript_text` with a new speaker named 'Omar'. Include one action item and one decision. Re-run and verify Omar appears in the attendees list.

2. **Add a new keyword:** Add `'responsible'` to `action_item_trigger_words` and add a transcript line like 'Sarah: James is responsible for the vendor contract.' Verify it appears in the action items.

3. **Summary stats:** After the summary prints, write code that calculates and prints a simple statistics line: 'X attendees, Y decisions, Z action items per person on average.'